# GNN Cloud Notebook: GCN-2 vs GCN-3 Multi-Seed Comparison

This notebook is configured for Google Colab using GitHub as the code source. It clones the repo into `/content`, keeps Google Drive optional, and runs a multi-seed comparison of only the two architecture variants that still matter after the first architecture sweep.

Compared models:
- `GCN-2 Control`: current best prior architecture baseline
- `GCN-3`: the only architecture variant that showed enough signal to justify multi-seed validation

This comparison keeps the following fixed across models:
- Adam with the baseline two-phase schedule
- weighted edges
- richer node features derived at load time
- graph-level summary features concatenated after graph pooling
- standardized regression targets during training

No new lattice sets are required.

In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules
print(f'Running in Colab: {IN_COLAB}')

if IN_COLAB:
    %pip -q install torch-geometric
else:
    print('Colab dependency install cell skipped.')

In [ ]:
REPO_URL = 'https://github.com/aadams2006/NSF-REU-Summer-26.git'
REPO_DIR = '/content/NSF-REU-Summer-26'

if IN_COLAB:
    import os
    if not os.path.isdir(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
    else:
        print(f'Repo already exists at {REPO_DIR}')
else:
    print('Git clone cell skipped outside Colab.')

In [ ]:
from datetime import datetime
from getpass import getpass
from pathlib import Path
import os
import sys

USE_DRIVE_FOR_DATA = False
SAVE_OUTPUTS_TO_DRIVE = True
PUSH_RESULTS_TO_GITHUB = False
PUSH_MODEL_TO_GITHUB = False
DRIVE_DATA_ROOT = '/content/drive/MyDrive/lattice_data'
DRIVE_OUTPUT_ROOT = '/content/drive/MyDrive/GCN_Cloud_Outputs_Architecture_Comparison_Multi_Seed'
GIT_RESULTS_SUBDIR = 'active_projects/voronoi_lattice_pipeline/gnn_prototype/GCN_Cloud_Outputs_Architecture_Comparison_Multi_Seed'
GIT_BRANCH = 'main'
GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN', '').strip()
GIT_COMMIT_USERNAME = os.environ.get('GIT_COMMIT_USERNAME', '').strip()
GIT_COMMIT_EMAIL = os.environ.get('GIT_COMMIT_EMAIL', '').strip()
RUN_STAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

if IN_COLAB:
    repo_root = Path(REPO_DIR).resolve()
else:
    repo_root = Path.cwd().resolve()

pipeline_root = repo_root / 'active_projects' / 'voronoi_lattice_pipeline'
module_dir = pipeline_root / 'gnn_prototype'
optimization_dir = module_dir / 'GCN_Optimization'
if not (module_dir / 'colab_gnn_stiffness_prototype.py').is_file():
    raise FileNotFoundError(f'Module not found at {module_dir}')
if not (optimization_dir / 'architecture_comparison_runner.py').is_file():
    raise FileNotFoundError(f'Runner not found at {optimization_dir}')

# Keep the notebook process rooted at the pipeline directory so any fallback
# path discovery inside shared helpers resolves correctly in Colab.
os.chdir(pipeline_root)

if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))
if str(optimization_dir) not in sys.path:
    sys.path.insert(0, str(optimization_dir))

if IN_COLAB and (USE_DRIVE_FOR_DATA or SAVE_OUTPUTS_TO_DRIVE):
    from google.colab import drive
    drive.mount('/content/drive')

if USE_DRIVE_FOR_DATA:
    if not IN_COLAB:
        raise RuntimeError('USE_DRIVE_FOR_DATA is only supported in Colab.')
    drive_data_root = Path(DRIVE_DATA_ROOT)
    train_root = drive_data_root / 'Randomness_Sweep'
    predict_root = drive_data_root / 'Lattice_Guess_Prediction_Input_Data'
else:
    train_root = pipeline_root / 'source_archives' / 'lattice_data' / 'Randomness_Sweep'
    predict_root = pipeline_root / 'datasets' / 'Lattice_Guess_Prediction_Input_Data'

if IN_COLAB and SAVE_OUTPUTS_TO_DRIVE:
    output_root = Path(DRIVE_OUTPUT_ROOT)
else:
    output_root = Path('/content/gnn_outputs_architecture_comparison_multi_seed') if IN_COLAB else pipeline_root / 'gnn_prototype' / 'outputs_architecture_comparison_multi_seed'

git_output_root = repo_root / GIT_RESULTS_SUBDIR
output_dir = output_root / f'run_{RUN_STAMP}'
per_model_output_root = output_dir / 'per_model'
output_dir.mkdir(parents=True, exist_ok=True)
(output_root / 'latest_run.txt').write_text(str(output_dir), encoding='utf-8')

if PUSH_RESULTS_TO_GITHUB:
    if not GITHUB_TOKEN:
        GITHUB_TOKEN = getpass('Enter GitHub token: ').strip()
    if not GIT_COMMIT_USERNAME:
        GIT_COMMIT_USERNAME = input('Enter Git commit username or display name: ').strip()
    if not GIT_COMMIT_EMAIL:
        GIT_COMMIT_EMAIL = input('Enter Git commit email (GitHub noreply or verified email): ').strip()

print(f'Repo root: {repo_root}')
print(f'Pipeline root: {pipeline_root}')
print(f'Working directory: {Path.cwd()}')
print(f'Train data: {train_root}')
print(f'Prediction data: {predict_root}')
print(f'Output root: {output_root}')
print(f'Current run dir: {output_dir}')
print(f'Git output root: {git_output_root}')
print(f'Push results to GitHub: {PUSH_RESULTS_TO_GITHUB}')
print(f'GitHub token loaded: {bool(GITHUB_TOKEN)}')
print(f'Git commit username loaded: {bool(GIT_COMMIT_USERNAME)}')
print(f'Git commit email loaded: {bool(GIT_COMMIT_EMAIL)}')

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shutil
import subprocess
from IPython.display import display

from architecture_comparison_runner import ArchitectureConfig, run_architecture_experiment


def build_aggregate_frame(summary_frame: pd.DataFrame) -> pd.DataFrame:
    metric_columns = [
        'Validation_R2',
        'Test_R2',
        'Test_RMSE',
        'Prediction_R2',
        'Prediction_RMSE',
    ]
    grouped = summary_frame.groupby('Architecture')[metric_columns].agg(['mean', 'std', 'min', 'max'])
    grouped.columns = [f'{metric}_{stat}' for metric, stat in grouped.columns]
    return grouped.reset_index()


def build_delta_frame(summary_frame: pd.DataFrame) -> pd.DataFrame:
    pivot = summary_frame.pivot(index='Seed', columns='Architecture_Key', values=['Test_R2', 'Test_RMSE', 'Prediction_R2'])
    delta_frame = pd.DataFrame({
        'Seed': pivot.index,
        'Delta_Test_R2_GCN3_minus_GCN2': pivot[('Test_R2', 'gcn3')] - pivot[('Test_R2', 'gcn2_control')],
        'Delta_Test_RMSE_GCN3_minus_GCN2': pivot[('Test_RMSE', 'gcn3')] - pivot[('Test_RMSE', 'gcn2_control')],
        'Delta_Prediction_R2_GCN3_minus_GCN2': pivot[('Prediction_R2', 'gcn3')] - pivot[('Prediction_R2', 'gcn2_control')],
    })
    return delta_frame.sort_values('Seed').reset_index(drop=True)


def plot_multi_seed_architecture_summary(summary_frame: pd.DataFrame, save_path: Path) -> None:
    figure, axes = plt.subplots(2, 2, figsize=(16, 10))
    axes = axes.ravel()
    metrics = [
        ('Test_R2', 'Test R2'),
        ('Test_RMSE', 'Test RMSE'),
        ('Prediction_R2', 'Prediction R2'),
        ('Validation_R2', 'Validation R2'),
    ]
    colors = {
        'gcn2_control': '#355070',
        'gcn3': '#b56576',
    }
    labels = {
        'gcn2_control': 'GCN-2 Control',
        'gcn3': 'GCN-3',
    }

    for axis, (metric_key, title) in zip(axes, metrics):
        for architecture_key in ('gcn2_control', 'gcn3'):
            plot_frame = summary_frame[summary_frame['Architecture_Key'] == architecture_key].sort_values('Seed')
            axis.plot(
                plot_frame['Seed'],
                plot_frame[metric_key],
                marker='o',
                linewidth=2,
                label=labels[architecture_key],
                color=colors[architecture_key],
            )
        axis.set_title(title)
        axis.set_xlabel('Seed')
        axis.grid(alpha=0.3)
        axis.legend()

    figure.tight_layout()
    figure.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.close(figure)


def plot_aggregate_comparison(aggregate_frame: pd.DataFrame, save_path: Path) -> None:
    figure, axes = plt.subplots(1, 3, figsize=(18, 5))
    plot_specs = [
        ('Test_R2_mean', 'Test R2 Mean', 'Test_R2_std'),
        ('Test_RMSE_mean', 'Test RMSE Mean', 'Test_RMSE_std'),
        ('Prediction_R2_mean', 'Prediction R2 Mean', 'Prediction_R2_std'),
    ]
    colors = ['#355070', '#b56576']
    for axis, (metric_key, title, error_key) in zip(axes, plot_specs):
        axis.bar(aggregate_frame['Architecture'], aggregate_frame[metric_key], yerr=aggregate_frame[error_key], color=colors, alpha=0.9, capsize=4)
        axis.set_title(title)
        axis.tick_params(axis='x', rotation=20)
        axis.grid(axis='y', alpha=0.3)
    figure.tight_layout()
    figure.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.close(figure)

In [ ]:
SEEDS = (11, 42, 73, 101, 202)
HIDDEN_DIM = 24
ARCHITECTURES = [
    ('gcn2_control', 'GCN-2 Control (Best Prior)'),
    ('gcn3', 'GCN-3'),
]

print(f'Seeds: {SEEDS}')
print(f'Hidden dim: {HIDDEN_DIM}')
for key, label in ARCHITECTURES:
    print(f' - {label}: {key}')

In [ ]:
experiment_results = {}
summary_rows = []

for seed in SEEDS:
    print(f'=== Seed {seed} ===')
    experiment_results[seed] = {}
    for architecture_name, architecture_label in ARCHITECTURES:
        print(f'Running {architecture_label} for seed {seed}')
        config = ArchitectureConfig(
            architecture_name=architecture_name,
            architecture_label=architecture_label,
            hidden_dim=HIDDEN_DIM,
            seed=seed,
            output_group='architecture_comparison_multi_seed',
        )
        result = run_architecture_experiment(
            config,
            train_root=train_root,
            predict_root=predict_root,
            output_root=per_model_output_root,
        )
        experiment_results[seed][architecture_name] = result

        row = result['summary_frame'].iloc[0].to_dict()
        row['Output_Dir'] = str(result['output_dir'])
        summary_rows.append(row)

summary_frame = pd.DataFrame(summary_rows).sort_values(['Seed', 'Architecture']).reset_index(drop=True)
summary_path = output_dir / 'architecture_multi_seed_summary.csv'
summary_frame.to_csv(summary_path, index=False)

print(f'Saved per-run summary to {summary_path}')
display(summary_frame)

In [ ]:
aggregate_frame = build_aggregate_frame(summary_frame)
aggregate_path = output_dir / 'architecture_multi_seed_aggregate.csv'
aggregate_frame.to_csv(aggregate_path, index=False)

delta_frame = build_delta_frame(summary_frame)
delta_path = output_dir / 'gcn3_vs_gcn2_per_seed_delta.csv'
delta_frame.to_csv(delta_path, index=False)

print(f'Saved aggregate summary to {aggregate_path}')
print(f'Saved per-seed delta summary to {delta_path}')
display(aggregate_frame)
display(delta_frame)

In [ ]:
multi_seed_plot_path = output_dir / 'architecture_multi_seed_metric_summary.png'
aggregate_plot_path = output_dir / 'architecture_multi_seed_aggregate_comparison.png'

plot_multi_seed_architecture_summary(summary_frame, save_path=multi_seed_plot_path)
plot_aggregate_comparison(aggregate_frame, save_path=aggregate_plot_path)

best_test_architecture = aggregate_frame.sort_values('Test_R2_mean', ascending=False).iloc[0]
best_prediction_architecture = aggregate_frame.sort_values('Prediction_R2_mean', ascending=False).iloc[0]
delta_summary = {
    'mean_delta_test_r2_gcn3_minus_gcn2': float(delta_frame['Delta_Test_R2_GCN3_minus_GCN2'].mean()),
    'mean_delta_test_rmse_gcn3_minus_gcn2': float(delta_frame['Delta_Test_RMSE_GCN3_minus_GCN2'].mean()),
    'mean_delta_prediction_r2_gcn3_minus_gcn2': float(delta_frame['Delta_Prediction_R2_GCN3_minus_GCN2'].mean()),
}
best_payload = {
    'best_mean_test_architecture': best_test_architecture.to_dict(),
    'best_mean_prediction_architecture': best_prediction_architecture.to_dict(),
    'delta_summary': delta_summary,
}
best_summary_path = output_dir / 'best_architecture_multi_seed_summary.json'
best_summary_path.write_text(json.dumps(best_payload, indent=2), encoding='utf-8')

print(f'Saved multi-seed plot to {multi_seed_plot_path}')
print(f'Saved aggregate comparison plot to {aggregate_plot_path}')
print(f'Saved best-architecture summary to {best_summary_path}')
print('Best mean test architecture:')
print(best_test_architecture[['Architecture', 'Test_R2_mean', 'Test_RMSE_mean', 'Prediction_R2_mean']])
print()
print('Best mean prediction architecture:')
print(best_prediction_architecture[['Architecture', 'Prediction_R2_mean', 'Test_R2_mean', 'Test_RMSE_mean']])
print()
print('Mean deltas for GCN-3 minus GCN-2:')
print(pd.Series(delta_summary))

In [ ]:
print(f'Parent run dir: {output_dir}')
parent_files = sorted(path.name for path in output_dir.iterdir() if path.is_file())
print('Parent-level files:')
for file_name in parent_files:
    print(f' - {file_name}')

for seed, seed_results in experiment_results.items():
    for architecture_name, result in seed_results.items():
        print(f' - seed {seed}, {architecture_name}: {result["output_dir"]}')

if PUSH_RESULTS_TO_GITHUB:
    if not IN_COLAB:
        raise RuntimeError('GitHub auto-push is only intended for the Colab clone workflow.')
    if not GITHUB_TOKEN:
        raise ValueError('Set GITHUB_TOKEN before enabling PUSH_RESULTS_TO_GITHUB.')

    git_run_dir = git_output_root / output_dir.name
    if git_run_dir.exists():
        shutil.rmtree(git_run_dir)
    shutil.copytree(output_dir, git_run_dir)
    if not PUSH_MODEL_TO_GITHUB:
        for model_path in git_run_dir.rglob('lattice_gnn_model.pt'):
            model_path.unlink()

    (git_output_root / 'latest_run.txt').write_text(str(git_run_dir.relative_to(repo_root)), encoding='utf-8')

    subprocess.run(['git', '-C', str(repo_root), 'config', 'user.name', GIT_COMMIT_USERNAME], check=True)
    subprocess.run(['git', '-C', str(repo_root), 'config', 'user.email', GIT_COMMIT_EMAIL], check=True)

    remote_url = subprocess.run(
        ['git', '-C', str(repo_root), 'remote', 'get-url', 'origin'],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    auth_url = remote_url.replace('https://', f'https://{GITHUB_TOKEN}@', 1)
    subprocess.run(['git', '-C', str(repo_root), 'remote', 'set-url', 'origin', auth_url], check=True)

    try:
        subprocess.run(['git', '-C', str(repo_root), 'add', str(git_run_dir), str(git_output_root / 'latest_run.txt')], check=True)
        diff_result = subprocess.run(
            ['git', '-C', str(repo_root), 'diff', '--cached', '--quiet'],
            check=False,
        )
        if diff_result.returncode == 0:
            print('No GitHub changes to commit.')
        else:
            commit_message = f'Add GCN-2 vs GCN-3 architecture multi-seed cloud results for {output_dir.name}'
            subprocess.run(['git', '-C', str(repo_root), 'commit', '-m', commit_message], check=True)
            subprocess.run(['git', '-C', str(repo_root), 'push', 'origin', GIT_BRANCH], check=True)
            print(f'Pushed results to GitHub under {git_run_dir.relative_to(repo_root)}')
    finally:
        subprocess.run(['git', '-C', str(repo_root), 'remote', 'set-url', 'origin', remote_url], check=True)

In [ ]:
if IN_COLAB and not SAVE_OUTPUTS_TO_DRIVE:
    from google.colab import files
    archive_path = '/content/gnn_outputs_architecture_comparison_multi_seed.zip'
    !cd /content && zip -qr gnn_outputs_architecture_comparison_multi_seed.zip gnn_outputs_architecture_comparison_multi_seed
    files.download(archive_path)
else:
    print(f'Outputs are in {output_dir}')